# Best Neighborhood in Pittsburgh: Final Ranking

To find the **best neighborhood in Pittsburgh**, we combined three WPRDC datasets onto a **0 to 100 scale** (where 100 is best) and averaged them into a **Final Score**:
1. **Playground Score:** More playground equipment = higher score.
2. **Safety Score:** Fewer police arrests = higher score.
3. **311 Upkeep Score:** Fewer 311 maintenance requests = higher score.

In [4]:
import io
import pandas as pd
import matplotlib.pyplot as plt

# Your three WPRDC dataset URLs
playgrounds_url = "https://data.wprdc.org/datastore/dump/47350364-44a8-4d15-b6e0-5f79ddff9367"
requests_311_url = "https://data.wprdc.org/datastore/dump/29462525-62a6-45bf-9b5e-ad2e1c06348d"
arrests_url = "https://data.wprdc.org/dataset/b62e8c3f-f40e-4b3f-9c60-82509bc397fb/resource/e419c20c-8df4-4729-830c-e49427a656e0/download/arrests_2024_2026.xlsx"

# Load the datasets (works in both JupyterLite and standard Jupyter)
try:
    from pyodide.http import open_url, pyfetch
    import micropip
    await micropip.install("openpyxl")
    playgrounds = pd.read_csv(open_url(playgrounds_url))
    requests_311 = pd.read_csv(open_url(requests_311_url), low_memory=False)
    response = await pyfetch(arrests_url)
    arrests = pd.read_excel(io.BytesIO(await response.bytes()))
except ModuleNotFoundError:
    playgrounds = pd.read_csv(playgrounds_url)
    requests_311 = pd.read_csv(requests_311_url, low_memory=False)
    arrests = pd.read_excel(arrests_url)

# Standardize column names to lowercase
playgrounds.columns = playgrounds.columns.str.lower().str.strip()
requests_311.columns = requests_311.columns.str.lower().str.strip()
arrests.columns = arrests.columns.str.lower().str.strip()

if "incidentneighborhood" in arrests.columns:
    arrests = arrests.rename(columns={"incidentneighborhood": "neighborhood"})

# 1. Playground Score (0-100, more equipment = higher score)
play = playgrounds.groupby("neighborhood").size().reset_index(name="equipment")
play["playground_score"] = (play["equipment"] / play["equipment"].max()) * 100

# 2. Safety Score (0-100, fewer arrests = higher score)
arrests = arrests[~arrests["neighborhood"].isin(["Outside County", "Outside State", "Outside City"])]
ar = arrests.groupby("neighborhood").size().reset_index(name="arrests")
ar["safety_score"] = 100 * (1 - (ar["arrests"] - ar["arrests"].min()) / (ar["arrests"].max() - ar["arrests"].min()))

# 3. 311 Upkeep Score (0-100, fewer requests = higher score)
req = requests_311.groupby("neighborhood").size().reset_index(name="requests")
req["score_311"] = 100 * (1 - (req["requests"] - req["requests"].min()) / (req["requests"].max() - req["requests"].min()))

# 4. Merge all three together and compute the Final Score
final_df = pd.merge(ar, req, on="neighborhood", how="inner")
final_df = pd.merge(final_df, play, on="neighborhood", how="left")
final_df["playground_score"] = final_df["playground_score"].fillna(0)

final_df["final_score"] = (final_df["playground_score"] + final_df["safety_score"] + final_df["score_311"]) / 3
final_df = final_df.sort_values(by="final_score", ascending=False).reset_index(drop=True)

# 5. Plot Top 10 Best Neighborhoods
top_10 = final_df.head(10).sort_values(by="final_score")
plt.barh(top_10["neighborhood"], top_10["final_score"], color="steelblue")
plt.title("Top 10 Best Neighborhoods in Pittsburgh (Final Score)")
plt.xlabel("Final Score (Out of 100)")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

# 6. Show the Final Ranking Table
final_df[["neighborhood", "final_score", "playground_score", "safety_score", "score_311"]].head(10).round(2)

<class 'pyodide.ffi.JsException'>: NetworkError: Failed to execute 'send' on 'XMLHttpRequest': Failed to load 'https://data.wprdc.org/datastore/dump/47350364-44a8-4d15-b6e0-5f79ddff9367'.

## Conclusion
* **Winner:** The top neighborhoods overall are those that balance strong playground access with low arrest numbers and low 311 complaints (such as **Squirrel Hill South**, **Beechview**, **Highland Park**, and **Swisshelm Park**).
* **Takeaway:** While **Central Oakland** is a personal favorite for college students due to campus proximity, residential neighborhoods score higher on our combined livability metric.